<a href="https://colab.research.google.com/github/sap-tarshi-ghosh/bank-marketing-2026/blob/main/code/bank_marketing_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

from imblearn.over_sampling import SMOTE, ADASYN

In [ ]:
df = pd.read_csv('/content/bank-additional-full.csv', sep=';')

In [ ]:

# 3. Drop Leakage Feature
df.drop('duration', axis=1, inplace=True)


# 4. Handle "unknown" Values
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].replace('unknown', df[col].mode()[0])


# 5. Encode Target
df['y'] = df['y'].map({'no': 0, 'yes': 1})


# 6. One-Hot Encoding
df = pd.get_dummies(df, drop_first=True)


# 7. Split Features & Target
X = df.drop('y', axis=1)
y = df['y']


# 8. Train-Test Split (IMPORTANT)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


# 9. Scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# 10. Sampling Techniques
smote = SMOTE(random_state=42)
adasyn = ADASYN(random_state=42)

X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
X_train_ad, y_train_ad = adasyn.fit_resample(X_train, y_train)


# 11. Models
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "MLP": MLPClassifier(hidden_layer_sizes=(100,), max_iter=300, random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}


# 12. Evaluation Metrics
scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}


# 13. Cross Validation Setup
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)


# 14. Function to Evaluate Models
def evaluate_models(X, y, label):
    print(f"\n===== {label} =====")

    for name, model in models.items():
        scores = cross_validate(model, X, y, cv=skf, scoring=scoring)

        print(f"\nModel: {name}")
        print(f"Accuracy:  {np.mean(scores['test_accuracy']):.4f}")
        print(f"Precision: {np.mean(scores['test_precision']):.4f}")
        print(f"Recall:    {np.mean(scores['test_recall']):.4f}")
        print(f"F1 Score:  {np.mean(scores['test_f1']):.4f}")
        print(f"ROC-AUC:   {np.mean(scores['test_roc_auc']):.4f}")


# 15. Run Experiments

# Baseline
evaluate_models(X_train, y_train, "Baseline (No Sampling)")

# SMOTE
evaluate_models(X_train_sm, y_train_sm, "SMOTE")

# ADASYN
evaluate_models(X_train_ad, y_train_ad, "ADASYN")


===== Baseline (No Sampling) =====

Model: Decision Tree
Accuracy:  0.8404
Precision: 0.3095
Recall:    0.3392
F1 Score:  0.3234
ROC-AUC:   0.6243

Model: Random Forest
Accuracy:  0.8923
Precision: 0.5424
Recall:    0.2837
F1 Score:  0.3720
ROC-AUC:   0.7685


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptro


Model: MLP
Accuracy:  0.8843
Precision: 0.4770
Recall:    0.2875
F1 Score:  0.3581
ROC-AUC:   0.7246


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [07:48:58] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [07:48:59] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [07:49:01] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [07:49:02] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:


Model: XGBoost
Accuracy:  0.8964
Precision: 0.5879
Recall:    0.2724
F1 Score:  0.3717
ROC-AUC:   0.7863

===== SMOTE =====

Model: Decision Tree
Accuracy:  0.9045
Precision: 0.9013
Recall:    0.9084
F1 Score:  0.9048
ROC-AUC:   0.9053

Model: Random Forest
Accuracy:  0.9402
Precision: 0.9487
Recall:    0.9308
F1 Score:  0.9397
ROC-AUC:   0.9819


...


Model: XGBoost
Accuracy:  0.9377
Precision: 0.9679
Recall:    0.9054
F1 Score:  0.9356
ROC-AUC:   0.9738

===== ADASYN =====

Model: Decision Tree
Accuracy:  0.9041
Precision: 0.9013
Recall:    0.9074
F1 Score:  0.9043
ROC-AUC:   0.9053

Model: Random Forest
Accuracy:  0.9418
Precision: 0.9490
Recall:    0.9337
F1 Score:  0.9413
ROC-AUC:   0.9832


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(



Model: MLP
Accuracy:  0.8883
Precision: 0.8843
Recall:    0.8935
F1 Score:  0.8888
ROC-AUC:   0.9545


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [08:14:50] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [08:14:52] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [08:14:53] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [08:14:55] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:


Model: XGBoost
Accuracy:  0.9365
Precision: 0.9697
Recall:    0.9012
F1 Score:  0.9342
ROC-AUC:   0.9739
